In [1]:
import pandas as pd
df = pd.read_parquet("../data/vacancies.parquet")
print(df.shape)
df.head(3)

(73, 25)


,id,has_ld,title,description,date_posted,valid_through,employment_type,company,city,ld_experience,...,address,employment,schedule,working_hours,skills,salary_source,salary_gross,salary_text,work_format,pay_frequency
0,123595600,True,Ведущий инженер Microsoft,NAUKA (Наука-Связь)\n- многофункциональный опе...,2026-10-02T13:23:26.443+03:00,2026-11-01T13:23:26+03:00,None,Наука-Связь,Москва,None,...,"Москва , Кожуховская , 2-й Южнопортовый проезд...",Полная занятость,5/2,8,"[Active Directory, IIS, Ms Exchange Server, Po...",NaN,None,NaN,NaN,NaN
1,129605311,True,Специалист информационной безопасности / Анали...,В Департаменте ИБ в Управлении мониторинга инц...,2026-10-16T16:05:22.783+03:00,2026-11-15T16:05:22+03:00,None,"Банк ПСБ, ИТ",Москва,None,...,"Москва , Бауманская , Комсомольская , Красносе...",Полная занятость,2/2 и другие варианты,Вечерние или ночные смены,None,text,False,от 100 000 ₽ за месяц на руки,на месте работодателя,два раза в месяц
2,133525885,True,Ведущий системный администратор/инженер,Обязанности:\nАдминистрирование ИТ инфраструкт...,2026-10-14T17:01:08.731+03:00,2026-10-23T17:01:08.726+03:00,None,Технологии и Информационные Системы,посёлок городского типа Заречье,None,...,"посёлок городского типа Заречье , Мещерская , ...",Полная занятость,5/2,8,"[Администрирование баз данных MySQL, MySql Wor...",text,True,от 350 000 до 450 000 ₽ за месяц до вычета нал...,на месте работодателя,два раза в месяц


In [2]:
has_salary = df["salary_from"].notna() | df["salary_to"].notna()
print("С зарплатой: ", has_salary.mean().round(2))
print(df.loc[has_salary, "currency"].value_counts(), "\n")
print(df.loc[has_salary, "salary_period"].value_counts(dropna=False), "\n")
print(df.loc[has_salary, "salary_gross"].value_counts(dropna=False), "\n")
print(df.loc[has_salary, "salary_source"].value_counts())

С зарплатой:  0.6
currency
RUR    43
USD     1
Name: count, dtype: int64 

salary_period
month    44
Name: count, dtype: int64 

salary_gross
False    23
True     21
Name: count, dtype: int64 

salary_source
text    44
Name: count, dtype: int64


In [3]:
import numpy as np
def net_monthly(gross):
    """До вычета -> на руки"""
    if gross <= 200_000:
        return gross * 0.87
    return 200_000 * 0.87 + (gross - 200_000) * 0.85

def build_target(row):
    f, t = row["salary_from"], row["salary_to"]
    if pd.notna(f) and pd.notna(t):
        value, kind = (f+t) / 2, "both"
    elif pd.notna(f):
        value, kind = f, "from_only"
    elif pd.notna(t):
        value, kind = t, "to_only"

    else:
        return pd.Series({"salary": np.nan, "range_type": None})

    if row["salary_gross"] == True:
        value = net_monthly(value)
    return pd.Series({"salary":value, "range_type":kind})

mask = has_salary & (df["currency"] == "RUR") & (df["salary_period"] == "month")

data = df[mask].copy()

data[["salary", "range_type"]] = data.apply(build_target, axis=1)

data["log_salary"] = np.log(data["salary"])

print(len(data))
data[["title", "salary_from", "salary_to", "salary_gross", "range_type", "salary", "log_salary"]]

43


,title,salary_from,salary_to,salary_gross,range_type,salary,log_salary
1,Специалист информационной безопасности / Анали...,100000.0,NaN,False,from_only,100000.000,11.512925
2,Ведущий системный администратор/инженер,350000.0,450000.0,True,both,344000.000,12.748397
3,Заместитель директора магазина,75000.0,120000.0,False,both,97500.000,11.487608
5,Системный администратор Windows,120000.0,200000.0,False,both,160000.000,11.982929
6,"Товаровед в даркстор (г. Москва Широкая улица,...",NaN,113000.0,True,to_only,98310.000,11.495881
9,Системный администратор Linux,NaN,200000.0,True,to_only,174000.000,12.066811
10,Заместитель управляющего магазином Thomas Munz...,80000.0,NaN,True,from_only,69600.000,11.150520
13,Администратор магазина (м.Новые Черёмушки),67650.0,103730.0,False,both,85690.000,11.358491
14,"Начальник смены (г Долгопрудный, ул Дирижабель...",NaN,111000.0,True,to_only,96570.000,11.478023
15,Начальник отдела/специалист отдела системного ...,130000.0,NaN,False,from_only,130000.000,11.775290


In [4]:
df.isna().mean().sort_values(ascending=False).round(2)

employment_type    1.00
ld_experience      1.00
pay_frequency      0.55
salary_to          0.55
salary_from        0.53
skills             0.41
salary_text        0.40
salary_gross       0.40
salary_source      0.40
salary_period      0.40
currency           0.40
address            0.12
working_hours      0.04
work_format        0.03
schedule           0.01
city               0.00
has_ld             0.00
company            0.00
experience         0.00
employment         0.00
valid_through      0.00
date_posted        0.00
description        0.00
title              0.00
id                 0.00
dtype: float64

In [5]:
df.loc[has_salary, ["salary_from", "salary_to"]].describe().round(0)

,salary_from,salary_to
count,34.0,33.0
mean,126224.0,183288.0
std,63835.0,128811.0
min,2000.0,84887.0
25%,83000.0,111000.0
50%,120000.0,135000.0
75%,150000.0,200000.0
max,350000.0,750000.0


In [6]:
mask = (
    df["salary_from"].notna()
    & df["salary_to"].notna()
    & df["currency"].eq("RUR")
    & df["salary_period"].eq("month")
)

ranges = df.loc[
    mask,
    ["title", "experience", "salary_from", "salary_to", "salary_gross"]
].copy()

ranges["range_width"] = ranges["salary_to"] - ranges["salary_from"]
ranges["salary_midpoint"] = (ranges["salary_from"] + ranges["salary_to"]) / 2

ranges.sort_values("salary_midpoint")

,title,experience,salary_from,salary_to,salary_gross,range_width,salary_midpoint
23,Старший кассир администратор,1–3 года,59780.0,84887.0,True,25107.0,72333.5
18,Администратор-кассир,не требуется,75000.0,90000.0,False,15000.0,82500.0
27,Администратор магазина (Vnukovo Outlet Village...,1–3 года,65000.0,100000.0,False,35000.0,82500.0
13,Администратор магазина (м.Новые Черёмушки),1–3 года,67650.0,103730.0,False,36080.0,85690.0
71,Заместитель директора минимаркета (улица Бажова),не требуется,83000.0,91000.0,True,8000.0,87000.0
70,Заместитель директора минимаркета (ул. Свободы),не требуется,83000.0,91000.0,True,8000.0,87000.0
69,Заместитель директора минимаркета,не требуется,80000.0,95000.0,True,15000.0,87500.0
3,Заместитель директора магазина,не требуется,75000.0,120000.0,False,45000.0,97500.0
26,Администратор в продуктовый магазин,1–3 года,100800.0,120000.0,True,19200.0,110400.0
31,Администратор магазина (м.Нахимовский проспект...,1–3 года,103000.0,120000.0,False,17000.0,111500.0


In [7]:
df[["title", "experience", "salary_from", "salary_to"]].to_string(index=False)

'                                                                              title   experience  salary_from  salary_to\n                                                          Ведущий инженер Microsoft      3–6 лет          NaN        NaN\n                              Специалист информационной безопасности / Аналитик SOC     1–3 года     100000.0        NaN\n                                            Ведущий системный администратор/инженер  более 6 лет     350000.0   450000.0\n                                                     Заместитель директора магазина не требуется      75000.0   120000.0\n                                                    Главный системный администратор      3–6 лет          NaN        NaN\n                                                    Системный администратор Windows      3–6 лет     120000.0   200000.0\n                                 Товаровед в даркстор (г. Москва Широкая улица, 31)     1–3 года          NaN   113000.0\n                       

In [8]:
it_keywords = (
    "системный администратор",
    "devops",
    "администратор баз данных",
    "dba",
    "1с",
    "erp",
    "информационной безопасности",
)

pattern = "|".join(it_keywords)

it_vacancies = df[
    df["title"].str.contains(pattern, case=False, na=False)
].copy()

print(f"IT-вакансий: {len(it_vacancies)} из {len(df)}")
it_vacancies[["title", "salary_from", "salary_to"]].to_string(index=False)

IT-вакансий: 43 из 73


'                                                                              title  salary_from  salary_to\n                              Специалист информационной безопасности / Аналитик SOC     100000.0        NaN\n                                            Ведущий системный администратор/инженер     350000.0   450000.0\n                                                    Главный системный администратор          NaN        NaN\n                                                    Системный администратор Windows     120000.0   200000.0\n                                   Системный администратор Senior IT-инфраструктура          NaN        NaN\n                                                      Системный администратор Linux          NaN   200000.0\n                                                     Администратор баз данных (DBA)          NaN        NaN\n                                                    Главный системный администратор          NaN        NaN\n                  

In [9]:
def classify_title(title):
    title = title.lower()

    if "devops" in title:
        return "DevOps"
    if "баз данных" in title or "dba" in title or "ms sql" in title:
        return "Базы данных"
    if "1с" in title or "1 с" in title or "erp" in title:
        return "1С / ERP"
    if "безопасности" in title or "сзи" in title:
        return "Информационная безопасность"
    if "системный администратор" in title:
        return "Системное администрирование"
    return "Другая IT-роль"

it_vacancies["category"] = it_vacancies["title"].apply(classify_title)
it_vacancies.groupby("category").agg(
    vacancies=("title", "count"),
    salary_from_filled=("salary_from", "count"),
    salary_to_filled=("salary_to", "count"),
)

,vacancies,salary_from_filled,salary_to_filled
category,,,
1С / ERP,7,5,3
DevOps,5,0,0
Базы данных,7,1,2
Информационная безопасность,3,2,1
Системное администрирование,21,11,10


In [10]:
it_vacancies["has_salary"] = (
    it_vacancies["salary_from"].notna()
    | it_vacancies["salary_to"].notna()
)

it_vacancies["has_salary_range"] = (
    it_vacancies["salary_from"].notna()
    & it_vacancies["salary_to"].notna()
)

it_vacancies.groupby("category").agg(
    vacancies=("title", "count"),
    with_any_salary=("has_salary", "sum"),
    with_full_range=("has_salary_range", "sum"),
)

,vacancies,with_any_salary,with_full_range
category,,,
1С / ERP,7,6,2
DevOps,5,0,0
Базы данных,7,3,0
Информационная безопасность,3,2,1
Системное администрирование,21,13,8


In [11]:
it_vacancies.columns.tolist()

['id',
 'has_ld',
 'title',
 'description',
 'date_posted',
 'valid_through',
 'employment_type',
 'company',
 'city',
 'ld_experience',
 'salary_from',
 'salary_to',
 'currency',
 'salary_period',
 'experience',
 'address',
 'employment',
 'schedule',
 'working_hours',
 'skills',
 'salary_source',
 'salary_gross',
 'salary_text',
 'work_format',
 'pay_frequency',
 'category',
 'has_salary',
 'has_salary_range']

In [12]:
it_vacancies.loc[
    it_vacancies["has_salary_range"],
    ["category", "title", "salary_from", "salary_to", "salary_gross"],
].sort_values(["category", "salary_from"])

,category,title,salary_from,salary_to,salary_gross
37,1С / ERP,Технический писатель (enterprise / системная д...,130000.0,200000.0,True
46,1С / ERP,Специалист 1 С/ERP,180000.0,180000.0,True
51,Информационная безопасность,Ведущий системный администратор / Ведущий спец...,220000.0,320000.0,False
24,Системное администрирование,Системный администратор,110000.0,120000.0,False
5,Системное администрирование,Системный администратор Windows,120000.0,200000.0,False
49,Системное администрирование,Системный администратор / Инженер техподдержки,130000.0,130000.0,False
28,Системное администрирование,Системный администратор (Middle),150000.0,220000.0,False
62,Системное администрирование,Системный администратор,150000.0,200000.0,False
66,Системное администрирование,Ведущий системный администратор / Ведущий спец...,150000.0,200000.0,False
25,Системное администрирование,Системный администратор (технический специалист),159400.0,159400.0,True


In [13]:
import numpy as np

it_vacancies["salary_info_type"] = np.select(
    [
        it_vacancies["salary_from"].notna() & it_vacancies["salary_to"].notna(),
        it_vacancies["salary_from"].notna(),
        it_vacancies["salary_to"].notna(),
    ],
    [
        "Указан диапазон",
        "Указана только нижняя граница",
        "Указана только верхняя граница",
    ],
    default="Зарплата не указана",
)

it_vacancies["salary_info_type"].value_counts()

salary_info_type
Зарплата не указана               19
Указан диапазон                   11
Указана только нижняя граница      8
Указана только верхняя граница     5
Name: count, dtype: int64

In [14]:
print("Полный набор:", len(df))
print("IT-подвыборка:", len(it_vacancies))

it_vacancies["salary_info_type"].value_counts()

Полный набор: 73
IT-подвыборка: 43


salary_info_type
Зарплата не указана               19
Указан диапазон                   11
Указана только нижняя граница      8
Указана только верхняя граница     5
Name: count, dtype: int64

In [15]:
it_vacancies.groupby("category").agg(
    vacancies=("title", "count"),
    with_salary=("has_salary", "sum"),
)

,vacancies,with_salary
category,,
1С / ERP,7,6
DevOps,5,0
Базы данных,7,3
Информационная безопасность,3,2
Системное администрирование,21,13


In [16]:
it_vacancies.loc[
    it_vacancies["category"].eq("DevOps"),
    ["title", "salary_text", "salary_from", "salary_to", "salary_source"],
].to_string(index=False)

'                                       title salary_text  salary_from  salary_to salary_source\nСистемный администратор (с функциями DevOps)         NaN          NaN        NaN           NaN\n         DevOps Engineer (Platform Engeneer)         NaN          NaN        NaN           NaN\n                              DevOps-инженер         NaN          NaN        NaN           NaN\n           DevOps-инженер (Астра Консалтинг)         NaN          NaN        NaN           NaN\n                    Ведущий инженер (DevOps)         NaN          NaN        NaN           NaN'